# EDGAR Financial Sentiment — Part 3: BERT `[CLS]` Classifier

Adapted from **Assignment 2, §3** (sentiment with a BERT `[CLS]` classification head), retargeted to finance.

**The question this part answers:** does a model pretrained on *financial* text give better sentence representations for sentiment? We put the **same** linear classification head on two **different** encoders — **(A)** `bert-base-cased` and **(B)** `ProsusAI/finbert` — fine-tune both on **Financial PhraseBank**, and compare accuracy. Only the backbone changes (our one variable). FinBERT is the *published, fully-trained* version of the Part-1/Part-2 idea, so this is the payoff test.

**What you'll practice:** the same four building blocks as Part 2 — a `Dataset`, an `nn.Module`, and the train / test loops — but with a genuinely different model shape: a **real classifier head** on `[CLS]`, class-index labels, and `CrossEntropyLoss` (no next-token trick this time).

> **Run in Google Colab with a T4 GPU.**

### The approach: classify off the `[CLS]` token

BERT is an **encoder**: every input token gets a contextual vector, and the special **`[CLS]`** token (position 0) is designed to carry a whole-sentence summary. The recipe:

1. Run the sentence through the encoder.
2. Take the `[CLS]` vector — `last_hidden_state[:, 0, :]`, shape `(batch, 768)`.
3. Feed it to a small `nn.Linear(768, 3)` head → 3 class logits.
4. Train end-to-end with `CrossEntropyLoss` against the class index (0/1/2).

**Contrast with Part 2:** there we padded *left* because a causal model reads the prediction off the *last* position. BERT reads off `[CLS]` at position **0**, so ordinary **right-padding is fine** — much simpler. We still pass an `attention_mask` so the encoder ignores the pad tokens.

## 0. Setup

In [1]:
# No datasets-version pin needed here: we load PhraseBank from its raw zip (Part 2's robust path)
# and never touch a script-based dataset.
!pip install -q -U transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00


In [2]:
import random, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset as HFDataset, ClassLabel   # aliased so it doesn't shadow torch's Dataset

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

Device: cuda


## 1. Config & seeds

In [4]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

max_len     = 64
batch_size  = 16
lr          = 2e-5      # standard full BERT fine-tuning rate
EPOCHS      = 3
NUM_CLASSES = 3

# the two backbones we compare: same architecture + head, different pretraining
MODELS = {
    'bert-base-cased': 'bert-base-cased',
    'FinBERT':         'ProsusAI/finbert',
}

## 2. Load the Financial PhraseBank benchmark

Same data and same stratified 80/20 split as Part 2, loaded straight from the original zip (no script loader). Reused here as-is — the new material is the model, below.

In [5]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')

_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)

pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=['negative', 'neutral', 'positive']))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']
label_names = pb_train.features['label'].names
print('Label index -> name:', dict(enumerate(label_names)))
print('Train size:', len(pb_train), '| Test size:', len(pb_test))
print('Example row:', pb_train[0])

Casting the dataset:   0%|          | 0/2264 [00:00<?, ? examples/s]

Label index -> name: {0: 'negative', 1: 'neutral', 2: 'positive'}
Train size: 1811 | Test size: 453
Example row: {'sentence': 'Sarantel , based in Wellingborough , UK , designs high-performance antennas for portable wireless devices .', 'label': 1}


## 3. Build the BERT classification `Dataset`  &larr; **your code**

Simpler than Part 2: just tokenize each sentence (right-padding is fine for BERT) and keep the class index as the label. For each row store:
- `input_ids` — token ids `(max_len,)`
- `attention_mask` — 1 for real tokens, 0 for padding `(max_len,)`
- `label` — the class index 0/1/2 (an `int64` scalar)

Keep the tensors on **CPU** here and move each batch to the GPU inside the loops (the scalable pattern). The `Dataset` takes the `tokenizer` as an argument because the two models use *different* tokenizers (cased vs. uncased), so we rebuild the data per model.

In [6]:
class BERTClassificationData(Dataset):
    """Tokenize (sentence, label) rows for a BERT-style encoder.

    BERT reads its sentence representation off the [CLS] token at position 0,
    so ordinary right-padding is fine (no left-padding like the GPT-2 part).
    Store, per row:
      'input_ids'      : token ids                        (max_len,)
      'attention_mask' : 1 for real tokens, 0 for padding (max_len,)
      'label'          : the class index 0/1/2            (scalar, int64)
    """
    def __init__(self, hf_split, tokenizer, max_len):
        self.data = []
        for ex in hf_split:
            ### YOUR CODE HERE
            # 1. tokenize ex['sentence'] with: max_length=max_len, truncation=True,
            #    padding='max_length', return_tensors='pt'  -> 'input_ids' & 'attention_mask'
            tokens = tokenizer(ex['sentence'], max_length=max_len, truncation=True,
                      padding='max_length', return_tensors='pt')

            # 2. append a dict to self.data (keep tensors on CPU, no .to(device)):
            #      'input_ids'      -> enc['input_ids'].squeeze(0)
            #      'attention_mask' -> enc['attention_mask'].squeeze(0)
            #      'label'          -> torch.tensor(ex['label'], dtype=torch.int64)
            self.data.append(
                {
                    'input_ids': tokens['input_ids'].squeeze(0),
                    'attention_mask': tokens['attention_mask'].squeeze(0),
                    'label': torch.tensor(ex['label'], dtype=torch.int64)
                }
            )
            ### END YOUR CODE

    def __len__(self):
        ### YOUR CODE HERE
        # return how many examples you stored
        return len(self.data)
        ### END YOUR CODE

    def __getitem__(self, index):
        ### YOUR CODE HERE
        # return the stored dict at position `index`
        return self.data[index]
        ### END YOUR CODE

### Sanity-check your `Dataset`
Run after filling in the cell above. The first token should decode to `[CLS]`, and the label index should match `label_names`.

In [7]:
_tok = AutoTokenizer.from_pretrained('bert-base-cased')
_probe = BERTClassificationData(pb_train, _tok, max_len)
print('Examples built:', len(_probe))

_ex = _probe[0]
print('input_ids shape:', tuple(_ex['input_ids'].shape), '| real tokens:', int(_ex['attention_mask'].sum()))
print('first token:', repr(_tok.decode([int(_ex['input_ids'][0])])), '(should be [CLS])')
print('label:', int(_ex['label']), '->', label_names[int(_ex['label'])])
print('Decoded (real tokens):', repr(_tok.decode(_ex['input_ids'][_ex['attention_mask'].bool()])))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Examples built: 1811
input_ids shape: (64,) | real tokens: 25
first token: '[CLS]' (should be [CLS])
label: 1 -> neutral
Decoded (real tokens): '[CLS] Sarantel, based in Wellingborough, UK, designs high - performance antennas for portable wireless devices. [SEP]'


## 4. Define the `[CLS]` classifier `nn.Module`  &larr; **your code**

This is the new shape vs. Part 2. The module holds two pieces:
- `self.encoder` — the BERT backbone (passed in);
- `self.classifier` — an `nn.Linear(hidden_size, num_classes)` head you create.

`forward` receives a batch dict, runs the encoder with the attention mask, pulls the `[CLS]` vector (`last_hidden_state[:, 0, :]`), and returns the head's logits `(batch, num_classes)`. Use `encoder.config.hidden_size` for the input width (768 for both models).

In [8]:
class BERTClassifier(nn.Module):
    """A BERT encoder + a linear classification head on the [CLS] token."""
    def __init__(self, encoder, num_classes):
        ### YOUR CODE HERE
        # 1. super().__init__()
        # 2. store the encoder as self.encoder
        # 3. create self.classifier = nn.Linear(encoder.config.hidden_size, num_classes)
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, num_classes)
        ### END YOUR CODE

    def forward(self, batch):
        ### YOUR CODE HERE
        # 1. run self.encoder(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        # 2. take the [CLS] vector: out.last_hidden_state[:, 0, :]   -> (batch, hidden)
        # 3. return self.classifier(<that>)                          -> (batch, num_classes)
        out = self.encoder(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        classification_token = out.last_hidden_state[:, 0, :]
        return self.classifier(classification_token)
        ### END YOUR CODE

loss_fn = nn.CrossEntropyLoss()

## 5. Train and test loops  &larr; **your code**

Both loops move each batch to the device first (since the `Dataset` stays on CPU). The **train loop** does one epoch of standard classification fine-tuning. The **test loop** reports a single accuracy: `argmax(logits) == label`.

In [9]:
def train_loop(dataloader, model, loss_fn, optimizer, reporting_interval=50, steps=None):
    """One epoch of fine-tuning.

    - model.train(); track running loss + batch count
    - for each batch (stop after `steps` if given):
        - move the batch dict to device:  {k: v.to(device) for k, v in batch.items()}
        - logits = model(batch)  -> (batch, num_classes); targets = batch['label']
        - zero_grad, CrossEntropyLoss, backward, optimizer.step()
        - accumulate loss; print running avg every `reporting_interval` batches
    """
    ### YOUR CODE HERE
    # follow the steps in the docstring above
    model.train()
    running_loss = 0.0
    batch_count = 0

    for batch in dataloader:
      if steps is not None and steps >= batch_count:
        break
      batch = {k: v.to(device) for k, v in batch.items()}
      logits = model(batch)
      targets = batch['label']

      optimizer.zero_grad()
      loss = loss_fn(logits, targets)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
      batch_count += 1

      if batch_count % reporting_interval == 0:
        print(f'step {batch_count}: avg loss = {(running_loss / batch_count):.4f}')

    print(f'Final Training Loss: {(running_loss / batch_count):.4f}')

    ### END YOUR CODE

In [10]:
def test_loop(dataloader, model, loss_fn, steps=None):
    """Evaluate accuracy: argmax(logits) == label.

    - model.eval(); wrap in torch.no_grad()
    - for each batch (stop after `steps`): move to device, get logits, add the loss,
      count correct (argmax == batch['label']) and total examples
    - print accuracy + avg loss; return accuracy in percent
    """
    ### YOUR CODE HERE
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    batch_count = 0

    with torch.no_grad():
      for batch in dataloader:
        if steps is not None and batch_count >= steps:
          break
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(batch)
        targets = batch['label']
        loss = loss_fn(logits, targets)

        running_loss += loss.item()
        batch_count += 1

        predictions = torch.argmax(logits, dim=1) # get the index of the biggest logit
        correct += (predictions == targets).sum().item()
        total += batch['label'].size(0)

    print(f'Final Test Loss: {(running_loss / batch_count):.4f}')
    print(f'Final Test Accuracy: {(correct / total):.4f}')
    return (correct / total) * 100
    ### END YOUR CODE

## 6. Run the comparison — bert-base vs. FinBERT

For each backbone: load its tokenizer + encoder, rebuild the loaders with that tokenizer, attach a fresh head, fine-tune for `EPOCHS`, and evaluate. `AutoModel` loads just the encoder backbone — for FinBERT it drops the published sentiment head (you'll see a warning about unused `classifier.*` weights; that's expected, because we train our own head for a fair architecture-matched comparison).

In [11]:
results = {}
for name, hf_id in MODELS.items():
    print(f'\n================ {name}  ({hf_id}) ================')
    tok = AutoTokenizer.from_pretrained(hf_id)
    encoder = AutoModel.from_pretrained(hf_id).to(device)   # backbone only

    train_loader = DataLoader(BERTClassificationData(pb_train, tok, max_len),
                              batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader  = DataLoader(BERTClassificationData(pb_test, tok, max_len),
                              batch_size=batch_size, shuffle=False)

    model = BERTClassifier(encoder, NUM_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(EPOCHS):
        print(f'-- epoch {epoch+1}/{EPOCHS} --')
        train_loop(train_loader, model, loss_fn, optimizer)
    print('Evaluating...')
    results[name] = test_loop(test_loader, model, loss_fn)

    del model, encoder
    torch.cuda.empty_cache()

print('\n================ RESULTS ================')
for name, acc in results.items():
    print(f'{name:18s} test accuracy: {acc:.1f}%')
if 'FinBERT' in results and 'bert-base-cased' in results:
    print(f'FinBERT payoff: {results["FinBERT"] - results["bert-base-cased"]:+.1f} pts')


================ bert-base-cased  (bert-base-cased) ================


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-- epoch 1/3 --
step 50: avg loss = 0.5387
step 100: avg loss = 0.3469
Final Training Loss: 0.3260
-- epoch 2/3 --
step 50: avg loss = 0.0387
step 100: avg loss = 0.0623
Final Training Loss: 0.0616
-- epoch 3/3 --
step 50: avg loss = 0.0151
step 100: avg loss = 0.0226
Final Training Loss: 0.0242
Evaluating...
Final Test Loss: 0.1585
Final Test Accuracy: 0.9514

================ FinBERT  (ProsusAI/finbert) ================


...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

-- epoch 1/3 --
step 50: avg loss = 0.2489
step 100: avg loss = 0.1754
Final Training Loss: 0.1610
-- epoch 2/3 --
step 50: avg loss = 0.0291
step 100: avg loss = 0.0276
Final Training Loss: 0.0262
-- epoch 3/3 --
step 50: avg loss = 0.0116
step 100: avg loss = 0.0165
Final Training Loss: 0.0159
Evaluating...
Final Test Loss: 0.1071
Final Test Accuracy: 0.9757

================ RESULTS ================
bert-base-cased    test accuracy: 95.1%
FinBERT            test accuracy: 97.6%
FinBERT payoff: +2.4 pts


## 7. What to look for

- **FinBERT &gt; bert-base-cased** is the expected, clean result: a backbone pretrained on financial text produces `[CLS]` representations that are already closer to what the sentiment task needs, so the same head learns a better classifier. This is the *published-model* version of the Part-1/Part-2 lesson — the same reason continued pretraining on EDGAR helped GPT-2.
- **Compare across parts.** Your Part-2 GPT-2 numbers were ~82% / ~86%. A fine-tuned BERT `[CLS]` classifier on PhraseBank-allagree typically lands in the high 80s to mid 90s — note where these two backbones fall and whether FinBERT closes or widens the gap.
- **Fair-comparison note:** we did *not* use FinBERT's off-the-shelf sentiment head — we trained our own on top of each backbone, so the only difference between A and B is the pretrained encoder weights. (Scoring FinBERT's *original* head zero-shot is a nice optional add-on, but it wouldn't be architecture-matched.)
- **Things to try:** freeze the encoder and train only the head (much faster — tests how good the *raw* features are); raise `EPOCHS`; or add FinBERT's zero-shot head as a third bar.

**Next:** Part 4 (QLoRA fine-tuning of a larger model via the HF `Trainer`) or jump to the **Part 8 bake-off** that scores every approach — lexicon, fine-tuned GPT-2, FinBERT — on one held-out table.

## 8. Optional explorations

Three add-ons building on the comparison above — run them after Section 6 (they reuse `BERTClassifier`, `BERTClassificationData`, `train_loop`, and `test_loop`). The first two each need one small function written; the third is just a config tweak.

### 8a. FinBERT zero-shot — using its *own* sentiment head

FinBERT already ships with a trained 3-class sentiment head, so we can classify with **no training at all**. Two gotchas:
- **Label order differs.** FinBERT's classes are `{0: positive, 1: negative, 2: neutral}` — *not* our `{0: negative, 1: neutral, 2: positive}`. Read `model.config.id2label` and remap, rather than hardcoding.
- **Leakage caveat.** FinBERT was fine-tuned *on Financial PhraseBank*, so this "zero-shot" score very likely includes sentences it trained on. Expect an inflated, **not directly comparable** number — that's the lesson, not a bug. It's a different bar (its own head), so it isn't architecture-matched to the §6 comparison.

In [33]:
def zero_shot_eval(dataloader, model, id2label, our_index):
    """Score a model that ALREADY has a sentiment head — no training.

    - model.eval(); torch.no_grad()
    - build a remap tensor: FinBERT class id -> OUR class index, using
      our_index[id2label[i].lower()]  for each FinBERT id i
    - per batch: move to device; logits = model(input_ids=..., attention_mask=...).logits;
      pred = argmax over logits; map pred through remap; count correct vs batch['label']
    - print + return accuracy %
    """
    ### YOUR CODE HERE
    # tip: remap = torch.tensor([our_index[id2label[i].lower()] for i in range(len(id2label))], device=device)
    model.eval()
    count = 0
    correct = 0
    with torch.no_grad():
      remap = torch.tensor([our_index[id2label[i].lower()] for i in range(len(id2label))], device=device)
      for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        count += batch['label'].size(0)
        logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
        predictions = torch.argmax(logits,dim=-1)
        correct += (remap[predictions] == batch['label']).sum().item()
        print(f'Accuracy: {(correct / count):.4%}')
    print(f'Final Accuracy: {(correct / count):.4%}')
    return (correct / count) * 100
    ### END YOUR CODE

In [34]:
from transformers import AutoModelForSequenceClassification

fin_tok   = AutoTokenizer.from_pretrained('ProsusAI/finbert')
fin_model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(device)
print('FinBERT id2label:', fin_model.config.id2label)   # note: NOT our label order!

our_index = {'negative': 0, 'neutral': 1, 'positive': 2}
fin_test_loader = DataLoader(BERTClassificationData(pb_test, fin_tok, max_len),
                             batch_size=batch_size, shuffle=False)
zero_shot_eval(fin_test_loader, fin_model, fin_model.config.id2label, our_index)

del fin_model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT id2label: {0: 'positive', 1: 'negative', 2: 'neutral'}
Accuracy: 100.0000%
Accuracy: 96.8750%
Accuracy: 97.9167%
Accuracy: 98.4375%
Accuracy: 98.7500%
Accuracy: 98.9583%
Accuracy: 99.1071%
Accuracy: 99.2188%
Accuracy: 97.9167%
Accuracy: 97.5000%
Accuracy: 97.7273%
Accuracy: 97.9167%
Accuracy: 97.5962%
Accuracy: 96.8750%
Accuracy: 96.6667%
Accuracy: 96.8750%
Accuracy: 96.6912%
Accuracy: 96.8750%
Accuracy: 96.7105%
Accuracy: 96.5625%
Accuracy: 96.7262%
Accuracy: 96.8750%
Accuracy: 97.0109%
Accuracy: 97.1354%
Accuracy: 97.0000%
Accuracy: 96.8750%
Accuracy: 96.9907%
Accuracy: 96.8750%
Accuracy: 96.9095%
Final Accuracy: 96.9095%


### 8b. Freeze the encoder — how good are the *raw* features?

Full fine-tuning updates the whole encoder. If we **freeze** it and train *only* the linear head, we're asking: how separable are the sentiment classes in the backbone's off-the-shelf `[CLS]` features? It's much faster (only ~2,300 trainable params) and usually **noticeably lower** accuracy — the gap from the ~95% full-fine-tune number tells you how much the encoder itself adapted during training.

In [35]:
def freeze_encoder(model):
    """Freeze the BERT backbone so only the classifier head trains."""
    ### YOUR CODE HERE
    # set requires_grad = False on every parameter of model.encoder
    for param in model.encoder.parameters():
      param.requires_grad = False
    ### END YOUR CODE

In [37]:
# Head-only training on bert-base-cased: how separable are the RAW (un-fine-tuned) features?
EPOCHS = 20
tok = AutoTokenizer.from_pretrained('bert-base-cased')
encoder = AutoModel.from_pretrained('bert-base-cased').to(device)
head_only = BERTClassifier(encoder, NUM_CLASSES).to(device)
freeze_encoder(head_only)

trainable = sum(p.numel() for p in head_only.parameters() if p.requires_grad)
total     = sum(p.numel() for p in head_only.parameters())
print(f'Trainable params (head only): {trainable:,} / {total:,}')

tr = DataLoader(BERTClassificationData(pb_train, tok, max_len), batch_size=batch_size, shuffle=True, drop_last=True)
te = DataLoader(BERTClassificationData(pb_test,  tok, max_len), batch_size=batch_size, shuffle=False)
opt = torch.optim.AdamW([p for p in head_only.parameters() if p.requires_grad], lr=2e-3)  # higher lr is fine for a head-only fit
for epoch in range(EPOCHS):
    print(f'-- epoch {epoch+1}/{EPOCHS} --')
    train_loop(tr, head_only, loss_fn, opt)
print('Evaluating (head-only)...')
test_loop(te, head_only, loss_fn)

del head_only, encoder
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params (head only): 2,307 / 108,312,579
-- epoch 1/20 --
step 50: avg loss = 0.6932
step 100: avg loss = 0.6102
Final Training Loss: 0.5907
-- epoch 2/20 --
step 50: avg loss = 0.4374
step 100: avg loss = 0.4228
Final Training Loss: 0.4190
-- epoch 3/20 --
step 50: avg loss = 0.3628
step 100: avg loss = 0.3716
Final Training Loss: 0.3754
-- epoch 4/20 --
step 50: avg loss = 0.3724
step 100: avg loss = 0.3610
Final Training Loss: 0.3630
-- epoch 5/20 --
step 50: avg loss = 0.3337
step 100: avg loss = 0.3365
Final Training Loss: 0.3339
-- epoch 6/20 --
step 50: avg loss = 0.2918
step 100: avg loss = 0.3075
Final Training Loss: 0.3138
-- epoch 7/20 --
step 50: avg loss = 0.3007
step 100: avg loss = 0.3066
Final Training Loss: 0.3071
-- epoch 8/20 --
step 50: avg loss = 0.3233
step 100: avg loss = 0.3100
Final Training Loss: 0.3025
-- epoch 9/20 --
step 50: avg loss = 0.2823
step 100: avg loss = 0.2972
Final Training Loss: 0.2945
-- epoch 10/20 --
step 50: avg loss = 0.2698
step 

### 8c. Raise `EPOCHS`

The simplest knob: bump `EPOCHS` in the config cell (Section 1) and re-run Section 6. With only ~1,800 training sentences, watch for test accuracy plateauing — or dipping (overfitting) — after a few epochs. (Your train loss already fell to ~0.02 by epoch 3, so there's likely little headroom left here.)